### Run shared notebook

In [0]:
dbutils.widgets.text('key_vault_scope', '', '')

In [0]:
key_vault_scope = dbutils.widgets.get('key_vault_scope') # pylint: disable=unused-variable

In [0]:
%run /dxcore/Utilities/MailAlerts $key_vault_scope=key_vault_scope

###Get parameters

In [0]:
import json
from typing import Dict, Tuple
#
# Capture input parameters
#
dbutils.widgets.text('landing_partition', '', '')
dbutils.widgets.text('params', '', '')

landing_partition = dbutils.widgets.get('landing_partition')  # pylint: disable=unused-variable
json_params = dbutils.widgets.get('params')

print(json_params)

###Parse JSON parameter payload

In [0]:
#
# Parse JSON from input parameter
#
try:
  parsed_params = json.loads(json_params)
  storage_account = parsed_params['SinkAzureStorageAccountName'] # pylint: disable=unused-variable
except Exception as e:
  
  dbutils.notebook.exit(get_error_payload(f"Unable to parse input parameters, Error: {str(e)}"))
  

In [0]:
%run /dxcore/Utilities/Utilities $storage_account=storage_account

In [0]:
print(parsed_params)

###DataFrame loading helper methods

In [0]:
def get_landing_path(parsed_params: Dict[str, str]):
  container = parsed_params['SinkFileSystem']
  storage_account_uri = parsed_params['SinkAzureStorageAccountName']
#   storage_account_name = storage_account_uri[storage_account_uri.index('/') + 2:]
  storage_account_name = storage_account_uri
#   landing_directory = 'landing/'+parsed_params['BronzeSource']

  if parsed_params['SinkDatasetType'] == 'adls':
    landing_directory = parsed_params['SinkLandingDirectory']+'/*.csv'
  else:
    landing_directory = parsed_params['SinkLandingDirectory']+'/*.'+parsed_params['SinkDatasetType']
 
  return f"abfss://{container}@{storage_account_name}.dfs.core.windows.net/{landing_directory}"

###Schema check code

In [0]:
def normalize_names_and_datatypes(schema_dictionary: Dict[str, str]):  # pylint: disable=R1710
  if schema_dictionary:
    return {k.lower():v.upper() for k,v in schema_dictionary.items()}

def get_dropped_columns(old_schema: Dict[str, str], new_schema: Dict[str, str]) -> Dict[str, str]:
  if not old_schema:
    return None
  if not new_schema:
    return old_schema
  return {k:old_schema[k] for k in set(old_schema).difference(set(new_schema))}

def get_new_columns(old_schema: Dict[str, str], new_schema: Dict[str, str]) -> Dict[str, str]:
  if not old_schema:
    return new_schema
  if not new_schema:
    return None
  return {k:new_schema[k] for k in set(new_schema).difference(set(old_schema))}

def get_changed_columns(old_schema: Dict[str, str], new_schema: Dict[str, str]) -> Dict[str, str]:
  if not old_schema or not new_schema:
    return None
  return {k:f"{old_schema[k]} -> {new_schema[k]}" for k in set(old_schema).intersection(set(new_schema)) if old_schema[k] != new_schema[k]}
    
def are_schemas_compatible(old_schema: Dict[str, str], new_schema: Dict[str, str]) -> Tuple[bool, str, str, str]:
  #
  # Convert column names to lower-case and datatypes to upper-case
  #
  old_schema = normalize_names_and_datatypes(old_schema)
  new_schema = normalize_names_and_datatypes(new_schema)
  #
  # Get dictionaries of new columns, dropped columns and changed columns
  #
  new_columns_dict     = get_new_columns(old_schema, new_schema)
  dropped_columns_dict = get_dropped_columns(old_schema, new_schema)
  changed_columns_dict = get_changed_columns(old_schema, new_schema)
  #
  # If there are new columns in the dataset that were not present in the old schema, we must fail the pipeline
  # because while merging these new columns into Delta Lake is feasible, they will precipitate downstream failures
  # in the Azure Synapse Analytics portion of the pipeline. Datatype changes for existing columns will cause the
  # Delta Lake portion of the pipeline to fail at the bronze zone stage. By contrast, columns dropped from the schema
  # will not cause any challenges because Delta will not drop the corresponding columns, allowing both Databricks
  # and Synapse to proceed with the pipeline unimpeded.
  #
  if old_schema and (new_columns_dict or changed_columns_dict):
    return (False,
            get_schema_str_from_dict(new_columns_dict),
            get_schema_str_from_dict(dropped_columns_dict),
            get_schema_str_from_dict(changed_columns_dict)
           )
  return (True,
            get_schema_str_from_dict(new_columns_dict),
            get_schema_str_from_dict(dropped_columns_dict),
            get_schema_str_from_dict(changed_columns_dict)
         )

###Load dataset from landing zone into Spark DataFrame

In [0]:
dataset_uri = get_landing_path(parsed_params)
file_format = parsed_params['SinkDatasetType'].lower()
table_name = parsed_params['DataMovementShortName'] # pylint: disable=unused-variable

if file_format in ('csv'):
  try:
    landing_df = read_csv_files(dataset_uri)
  except Exception as e:
    dbutils.notebook.exit(get_error_payload(f"Unable to load CSV file from {dataset_uri}, Error: {str(e)}"))
    
# Adding if condition below for product and store tables for DG and WF to prevent pyspark from casting the table columns into different data types 
# by setting the value for the inferschema parameter as false while reading the dataframe
if file_format == 'gz':
  try:
    if table_name.__contains__("dg_product") or table_name.__contains__("wf_product") or table_name.__contains__("dg_store") or table_name.__contains__("wf_store"):
      infer_schema = 'false'
    else:
      infer_schema = 'true'
    landing_df = read_gz_files(dataset_uri, infer_schema)
  except Exception as e:
    dbutils.notebook.exit(get_error_payload(f"Unable to load GZ file from {dataset_uri}, Error: {str(e)}"))
    
elif file_format == 'adls':
  try:
    landing_df = read_csv_files(dataset_uri)
  except Exception as e:
    
    dbutils.notebook.exit(get_error_payload(f"Unable to load CSV file from {dataset_uri}, Error: {str(e)}"))
    
elif file_format == 'parquet':
  try:
    landing_df = get_dataset_parquet(dataset_uri)
  except Exception as e:
    
    dbutils.notebook.exit(get_error_payload(f"Unable to load parquet file from {dataset_uri}, Error: {str(e)}"))
    
elif file_format == 'orc':
  try:
    landing_df = get_dataset_orc(dataset_uri)
  except Exception as e:
    
    dbutils.notebook.exit(get_error_payload(f"Unable to load ORC file from {dataset_uri}, Error: {str(e)}"))
    
elif file_format == 'text':
  try:
    landing_df = get_dataset_delimited(
      uri=dataset_uri
      ,sep=parsed_params['SinkColumnDelim']
      ,quote=parsed_params['SinkQuoteChar']
      ,escape=parsed_params['SinkEscapeChar']
      ,header='True'
    )
  except Exception as e:
    
    dbutils.notebook.exit(get_error_payload(f"Unable to load delimited file from {dataset_uri}, Error: {str(e)}"))  
    

###Check whether schemas are compatible

In [0]:
#
# Get row count
#
try:
  row_count = landing_df.count()
except Exception as e:
  
  dbutils.notebook.exit(get_error_payload(f"Unable to get row count from dataframe, Error: {str(e)}"))
  
#
# Get schema from DataFrame
#
try:
  new_schema = get_schema_from_df(landing_df)
except Exception as e:
  
  dbutils.notebook.exit(get_error_payload(f"Unable to get schema from dataframe, Error: {str(e)}"))
  
#
# If created_dtm is not in the schema, add it
#
if new_schema.get('created_dtm', '').lower() != 'timestamp':
  new_schema['created_dtm'] = 'TIMESTAMP'
#
# If job_id is not in the schema, add it
#
if new_schema.get('job_id', '').lower() != 'string':
  new_schema['job_id'] = 'STRING'
#
# Get old schema from control table (received as a parameter)
#
old_schema = get_schema_dict_from_str(parsed_params['Schema'])
#
# Get schema compatibility flag and list of new columns, dropped columns and updated columns
#
are_schemas_compatible_flag, columns_added, columns_dropped, columns_updated = are_schemas_compatible(old_schema, new_schema)
#
# Print results
#
print(f"Old schema: {old_schema}\n")
print(f"New schema: {new_schema}\n")
print(f"Columns added: {columns_added}\n")
print(f"Columns dropped: {columns_dropped}\n")
print(f"Columns updated: {columns_updated}\n")

### Get MAX and MIN of audit columns

In [0]:
# sc.setJobDescription(f"Get latest timestamp for {dataset.table_name} dataset from DataFrame's audit columns")
  
# print(f'Latest timestamp in dataframe: {max_ts}')
max_ts = str(datetime.datetime.now())
min_ts = str(datetime.datetime.now())

### Returns success message to caller (in this case, Azure Data Factory)

In [0]:
dbutils.notebook.exit(json.dumps({
  "status": PipelineStatus.SUCCESS.value
  ,"are_schemas_compatible_flag": are_schemas_compatible_flag
  ,"columns_added" : columns_added
  ,"columns_dropped": columns_dropped
  ,"columns_updated": columns_updated
  ,"schema_str": get_schema_str_from_dict(new_schema)
  , "max_watermark_ts": max_ts if max_ts else None
  ,"min_watermark_ts": min_ts if min_ts else None
  ,"row_count": row_count
}))